# Team TRADON: Crowdedness Estimation Model for Delhi Buses
Transport Stack Hackathon 2026 — Problem 2 (Crowdedness-Aware Journey Planner)

**Objective:** In this notebook, I use October 2026 Delhi bus ticketing records and static GTFS schedule data to estimate bus crowding stretch by stretch across time bands and day types for our journey planner.

**Input Files Used:** I loaded `October-Delhi.csv`, `routes.txt`, `trips.txt`, `stops.txt`, and `stop_times.txt` into `/content` for processing.

I structured the pipeline sequentially from GTFS schedule parsing and stop matching to stretch-level demand aggregation, crowding classification, and sanity checks.

In [ ]:
import re, difflib
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

DATA_DIR = "/content"        # directory containing GTFS and October 2026 ticket data
FUZZY_CUTOFF = 0.65          # similarity threshold (0 to 1) for same-route stop name matching
MIN_TICKETS = 30             # minimum ticket threshold for reliable confidence

# Time bands defined for commuter and operational analysis
BANDS = {
    "night (0-5)": range(0, 5),
    "early (5-8)": range(5, 8),
    "morning peak (8-11)": range(8, 11),
    "midday (11-16)": range(11, 16),
    "evening peak (16-21)": range(16, 21),
    "late (21-24)": range(21, 24),
}
hour_to_band = {h: b for b, hrs in BANDS.items() for h in hrs}
norm = lambda s: re.sub(r"[^a-z0-9]", "", str(s).lower())   # normalize stop names for matching

## 1. Load GTFS Schedule Data
I load the static GTFS tables (`routes.txt`, `trips.txt`, `stops.txt`, `stop_times.txt`) to map routes, trips, stops, stop sequences, and scheduled departure hours.

In [ ]:
routes = pd.read_csv(f"{DATA_DIR}/routes.txt", usecols=["route_id", "route_short_name"])
trips = pd.read_csv(f"{DATA_DIR}/trips.txt", usecols=["route_id", "trip_id"])
stops = pd.read_csv(f"{DATA_DIR}/stops.txt", usecols=["stop_id", "stop_name"])
st = pd.read_csv(f"{DATA_DIR}/stop_times.txt",
                 usecols=["trip_id", "stop_id", "stop_sequence", "departure_time"])

# Combine GTFS tables into a single stop-visit table
st = st.merge(trips, on="trip_id").merge(routes, on="route_id").merge(stops, on="stop_id")
st["hour"] = st.departure_time.str.slice(0, 2).astype(int) % 24   # wrap post-midnight GTFS hours (e.g., 25:xx) modulo 24
st = st.sort_values(["trip_id", "stop_sequence"])
print(len(routes), "routes,", len(trips), "trips,", len(st), "stop visits")

## 2. Canonical Stop Sequence per Route
Because tickets record origin and destination stops rather than every intermediate stop, I construct a canonical ordered stop list per route using its longest scheduled trip. Directional routes (such as `UP` and `DOWN` variants) appear as distinct route identifiers in the dataset.

In [ ]:
n_stops = st.groupby(["route_short_name", "trip_id"]).size().rename("n").reset_index()
longest = n_stops.sort_values("n", ascending=False).drop_duplicates("route_short_name")

canon = st.merge(longest[["route_short_name", "trip_id"]], on=["route_short_name", "trip_id"])
canon = canon.sort_values(["route_short_name", "stop_sequence"])
canon["idx"] = canon.groupby("route_short_name").cumcount()     # 0-based stop index along the canonical route
canon["nname"] = canon.stop_name.map(norm)

# Stretch j represents the segment between stop index j and stop index j+1
canon[canon.route_short_name == "717AUP"][["idx", "stop_name"]].head(8)

## 3. Scheduled Supply by Stretch and Hour
I align scheduled trips to the canonical stop indices and count unique trips traversing each consecutive stop-to-stop stretch per hour.

In [ ]:
canon_idx = canon.set_index(["route_short_name", "stop_id"]).idx
canon_idx = canon_idx[~canon_idx.index.duplicated(keep="first")]

st["idx"] = pd.MultiIndex.from_arrays([st.route_short_name, st.stop_id]).map(canon_idx.to_dict())
st = st.dropna(subset=["idx"]).copy()
st["idx"] = st.idx.astype(int)
st["next_idx"] = st.groupby("trip_id").idx.shift(-1)

seg = st[st.next_idx == st.idx + 1]        # retain consecutive stop stretches only
supply = (seg.groupby(["route_short_name", "idx", "hour"]).trip_id.nunique()
             .rename("trips").reset_index()
             .rename(columns={"route_short_name": "route", "idx": "stretch"}))
supply.head()

## 4. Load October 2026 Ticketing Data
I load the October 2026 ticket dataset and filter to routes present in our GTFS canonical stop table.

In [ ]:
tk = pd.read_csv(f"{DATA_DIR}/October-Delhi.csv",
                 usecols=["passenger_count", "route", "start_stop_name", "end_stop_name", "created_at"])
n_all = len(tk)
tk = tk[tk.route.isin(canon.route_short_name.unique())].copy()
print(n_all, "tickets;", len(tk), "are on routes that exist in the GTFS")

## 5. Map Ticket Stop Names to Canonical Route Stops
Because ticket stop names can differ slightly in spelling or punctuation from GTFS stop names, I first match normalized strings exactly and fall back to same-route fuzzy matching at `FUZZY_CUTOFF = 0.65`.

In [ ]:
patterns = {r: dict(zip(g.nname, g.idx)) for r, g in canon.groupby("route_short_name")}
cache = {}

def find_idx(route, name):
    key = (route, name)
    if key in cache:
        return cache[key]
    n = norm(name)
    pat = patterns[route]
    if n in pat:
        res = pat[n]
    else:
        m = difflib.get_close_matches(n, pat.keys(), n=1, cutoff=FUZZY_CUTOFF)
        res = pat[m[0]] if m else -1       # -1 indicates unmatched stop name
    cache[key] = res
    return res

# Deduplicate (route, stop_name) pairs before matching for speed
pairs = pd.concat([tk[["route", "start_stop_name"]].set_axis(["route", "nm"], axis=1),
                   tk[["route", "end_stop_name"]].set_axis(["route", "nm"], axis=1)]).drop_duplicates()
for r, nm in zip(pairs.route, pairs.nm):
    find_idx(r, nm)

tk["s"] = [cache[(r, n)] for r, n in zip(tk.route, tk.start_stop_name)]
tk["e"] = [cache[(r, n)] for r, n in zip(tk.route, tk.end_stop_name)]

In [ ]:
usable = (tk.s >= 0) & (tk.e >= 0) & (tk.s < tk.e)     # valid matched origin and destination with forward progression
print(f"usable tickets: {usable.mean():.1%}")
print(f"start stop not found: {(tk.s < 0).mean():.1%}, end stop not found: {(tk.e < 0).mean():.1%}")
tk = tk[usable].copy()

## 6. Extract Time-of-Day and Day-Type Features
I parse ticket timestamps to extract the hour and classify days into `weekday` vs. `weekend`, counting observed days per category to compute daily averages.

In [ ]:
ts = pd.to_datetime(tk.created_at.str.slice(0, 19))
tk["hour"] = ts.dt.hour
tk["daytype"] = np.where(ts.dt.dayofweek >= 5, "weekend", "weekday")

days = pd.Series(ts.dt.normalize().unique())
n_days = {"weekday": int((days.dt.dayofweek < 5).sum()), "weekend": int((days.dt.dayofweek >= 5).sum())}
print(n_days)

## 7. Compute Stretch-Level Passenger Demand
To compute cumulative onboard riders and ticket counts across each stretch without looping over every stop pair, I add passenger counts/tickets at boarding index `s`, subtract at alighting index `e`, and take a cumulative sum along the route.

In [ ]:
max_idx = canon.groupby("route_short_name").idx.max().to_dict()
rows = []
for (route, dt_, hr), g in tk.groupby(["route", "daytype", "hour"]):
    L = int(max_idx[route]) + 2
    add = np.zeros(L); cnt = np.zeros(L)
    np.add.at(add, g.s.values, g.passenger_count.values)
    np.add.at(add, g.e.values, -g.passenger_count.values)
    np.add.at(cnt, g.s.values, 1)
    np.add.at(cnt, g.e.values, -1)
    riders = np.cumsum(add)[: L - 1]
    tickets = np.cumsum(cnt)[: L - 1]
    for j in range(L - 1):
        if riders[j] > 0:
            rows.append((route, j, hr, dt_, riders[j], tickets[j]))

demand = pd.DataFrame(rows, columns=["route", "stretch", "hour", "daytype", "riders", "tickets"])
demand["band"] = demand.hour.map(hour_to_band)
print(len(demand), "rows")

## 8. Calculate Crowding Index by Time Band
I aggregate daily average ticketed riders and scheduled trips into time bands, defining `crowd_index` as daily ticketed riders divided by scheduled trips on that stretch and band, and flagging slices with fewer than `MIN_TICKETS` as low confidence.

In [ ]:
supply["band"] = supply.hour.map(hour_to_band)
supply_band = supply.groupby(["route", "stretch", "band"]).trips.sum().reset_index()

dem_band = demand.groupby(["route", "stretch", "band", "daytype"])[["riders", "tickets"]].sum().reset_index()
dem_band["riders_per_day"] = dem_band.riders / dem_band.daytype.map(n_days)

out = dem_band.merge(supply_band, on=["route", "stretch", "band"], how="left")
out["crowd_index"] = out.riders_per_day / out.trips
out["low_confidence"] = out.tickets < MIN_TICKETS

print("rows:", len(out))
print("no schedule found for", f"{out.trips.isna().mean():.1%}", "of rows")
print("low confidence:", f"{out.low_confidence.mean():.1%}")

## 9. Map Crowding Index to Commuter Labels
I translate continuous crowding scores into intuitive labels (`Seats available`, `Standing room`, `Crowded`) using empirical quantiles (`0.50` and `0.85`) over reliable observations, and attach readable origin/destination stop names for each stretch.

In [ ]:
good = out[out.crowd_index.notna() & ~out.low_confidence].crowd_index
SEATS_UP_TO, STANDING_UP_TO = good.quantile([0.50, 0.85])
print(f"Seats available: up to {SEATS_UP_TO:.2f} | Standing room: up to {STANDING_UP_TO:.2f} | Crowded: above")

out["level"] = np.select(
    [out.crowd_index.isna(), out.crowd_index <= SEATS_UP_TO, out.crowd_index <= STANDING_UP_TO],
    ["No data", "Seats available", "Standing room"], "Crowded")

# Attach human-readable from_stop and to_stop labels
names = canon[["route_short_name", "idx", "stop_name"]].set_axis(["route", "stretch", "from_stop"], axis=1)
out = out.merge(names, on=["route", "stretch"], how="left")
nxt = names.assign(stretch=names.stretch - 1).rename(columns={"from_stop": "to_stop"})
out = out.merge(nxt, on=["route", "stretch"], how="left")
out[["route", "from_stop", "to_stop", "band", "daytype", "crowd_index", "level", "low_confidence"]].head(10)

## 10. Sanity Checks and Pattern Inspection
I run visual and tabular checks across time bands, weekdays vs. weekends, and high-crowding weekday evening peak stretches to verify that the model reflects expected transit behavior.

In [ ]:
v = out[out.crowd_index.notna() & ~out.low_confidence]
order = list(BANDS.keys())

fig, ax = plt.subplots(1, 2, figsize=(12, 4))
v.groupby("band").crowd_index.mean().reindex(order).plot.bar(ax=ax[0], title="Average crowding by time band")
v.groupby(["band", "daytype"]).crowd_index.mean().unstack().reindex(order).plot.bar(ax=ax[1], title="Weekday vs weekend")
plt.tight_layout(); plt.show()

In [ ]:
# Inspect top weekday evening-peak stretches among reliable rows
top = (v[(v.band == "evening peak (16-21)") & (v.daytype == "weekday")]
       .sort_values("crowd_index", ascending=False)
       [["route", "from_stop", "to_stop", "crowd_index", "tickets"]].head(10))
top

## 11. Export Crowding Lookup Table
I save the resulting stretch- and band-level crowding dataset to `crowding_by_band.csv` for use by the journey planner application.

In [ ]:
out.to_csv(f"{DATA_DIR}/crowding_by_band.csv", index=False)
print("saved crowding_by_band.csv")

## Summary of Findings, Assumptions, and Limitations (Team TRADON)
- **Temporal & Day-Type Patterns:** Crowding rises noticeably during commuter peak windows (`morning peak 8-11` and `evening peak 16-21`) compared to midday/off-peak hours, with distinct weekday commuting profiles versus weekend travel.
- **Matching & Threshold Sensitivity:** Tightening `FUZZY_CUTOFF` (e.g., to `0.8`) reduces false stop matches while lowering usable ticket coverage; adjusting quantile cut-offs (`0.50` / `0.85`) shifts how strictly stretches are classified into `Seats available`, `Standing room`, and `Crowded`.
- **Reliability & Caveats:** Estimates are least reliable on low-ticket slices (`tickets < MIN_TICKETS`), early/late off-peak bands, unmatched stop names, routes where pass holders or unrecorded riders are common, and routes where bus dimensions (`12 m` vs `9 m`) differ—and cover October 2026 rather than other seasons or festivals.